In [ ]:
%%writefile main.cpp
#include <iostream>
#include <vector>
#include <string>
#include <unordered_set>
#include <unordered_map>
#include <iomanip>
#include <limits>
using namespace std;


class Cart;
class Product;
class User;
class Marketplace;

class Product
{
private:
    int id;
    int sellerId;
    string name;
    double price;
    int stock;

public:
    Product(int pid, int psellerId, string pname, double pprice, int pstock)
    {
      id = pid;
      sellerId = psellerId;
      name = pname;
      price = pprice;
      stock = pstock;
    }

    int getId() { return id; }
    int getSellerId() { return sellerId; }
    string getName() { return name; }
    double getPrice() { return price; }
    int getStock() { return stock; }

    void setPrice(double newPrice) { price = newPrice; }
    void decreaseStock(int quant) { stock -= quant; }

    // check if item is available
    bool inStock() { return stock > 0; }
};

class Cart
{
private:
    unordered_map<int, int> items; // productId -> quantity

public:
    void addProduct(int productId, int quantity) { items[productId] += quantity; }
    void removeProduct(int productId) { items.erase(productId); }
    bool hasProduct(int productId) { return items.count(productId); }
    bool empty() { return items.empty(); }
    void clear() { items.clear(); }

    void updateQuantity(int productId, int quantity)
    {
        if (quantity <= 0)
            items.erase(productId);
        else
            items[productId] = quantity;
    }

    unordered_map<int, int>& getItems() { return items; }

    double getTotalPrice(vector<Product>& products)
    {
        double total = 0;
        for (auto item : items) {
            int productId = item.first;
            int quantity = item.second;
            total += products[productId].getPrice() * quantity;
        }
        return total;
    }
};

class User
{
private:
    int id;
    string username;
    string password;
    double balance;
    unordered_set<int> following;
    vector<int> listings;
    Cart cart;

public:
    User(int pid, string uname, string pwd, double money)
    {
      id = pid;
      username = uname;
      password = pwd;
      balance = money;
    }

    int getId() { return id; }
    string getUsername() { return username; }
    double getBalance() { return balance; }
    Cart& getCart() { return cart; }

    bool checkPassword(string pwd) { return password == pwd; }

    void follow(int userId) { following.insert(userId); }
    void unfollow(int userId) { following.erase(userId); }
    bool isFollowing(int userId) { return following.count(userId); }

    void addBalance(double amount) { balance += amount; }
    void addListing(int productId) { listings.push_back(productId); }
    vector<int>& getListings() { return listings; }

    bool pay(double amount)
    {
      if (balance < amount) {
        cout << "Insufficient balance!\n";
        return false;
      }
      balance -= amount;
      return true;
    }
};

class Marketplace
{
private:
    vector<User> users;
    vector<Product> products;
    unordered_map<string, int> usernameToIndex;
    int nextUserId = 1;
    User* currentUser;

public:
    Marketplace() { currentUser = nullptr; }

    void registerUser(string username, string password, double balance)
    {
      // cant have duplicate usernames
      if (usernameToIndex.count(username)) {
        cout << "Username already taken!\n";
        return;
      }
      User newUser(nextUserId++, username, password, balance);
      users.push_back(newUser);
      usernameToIndex[username] = users.size() - 1;
      cout << username << " registered successfully!\n";
    }

    bool login(string username, string password)
    {
      if (!usernameToIndex.count(username)) {
        cout << "User does not exist!\n";
        return false;
      }
      int index = usernameToIndex[username];
      if (!users[index].checkPassword(password)) {
        cout << "Wrong password!\n";
        return false;
      }
      currentUser = &users[index];
      cout << "Welcome " << username << "!\n";
      return true;
    }

    // helper to get name from id
    string getUsernameById(int id)
    {
      for (User &user : users)
        if (user.getId() == id) return user.getUsername();
      return "";
    }
    void browseMyListings()
{
    if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
    }

    cout << "\n================ MY LISTINGS ================\n\n";

    cout << left
         << setw(6) << "ID"
         << setw(18) << "Product"
         << setw(12) << "Price"
         << setw(10) << "Stock"
         << endl;

    cout << "---------------------------------------------\n";

    vector<int> &listings = currentUser->getListings();

    if (listings.empty()) {
        cout << "No products listed.\n";
        return;
    }

    for (int productId : listings) {
        Product &product = products[productId];

        cout << left
             << setw(6) << product.getId()
             << setw(18) << product.getName()
             << setw(12) << product.getPrice()
             << setw(10) << product.getStock()
             << endl;
    }
}

    void logout()
    {
      if (currentUser == nullptr) {
        cout << "No user is logged in!\n";
        return;
      }
      cout << currentUser->getUsername() << " logged out!\n";
      currentUser = nullptr;
    }

    void listProduct(string name, double price, int stock)
    {
      if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
      }
      int productId = products.size();
      Product newProduct(productId, currentUser->getId(), name, price, stock);
      products.push_back(newProduct);
      currentUser->addListing(productId);
      cout << name << " listed successfully!\n";
    }

    void changePrice(int productId, double newPrice)
    {
      if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
      }
      if (productId < 0 || productId >= products.size()) {
        cout << "Invalid product ID!\n";
        return;
      }
      Product &product = products[productId];
      // only owner can change price
      if (product.getSellerId() != currentUser->getId()) {
        cout << "You can only modify your own products!\n";
        return;
      }
      product.setPrice(newPrice);
      cout << "Price updated successfully!\n";
    }

    void addToCart(int productId, int quantity)
    {
      if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
      }
      if (productId < 0 || productId >= products.size()) {
        cout << "Invalid product!\n";
        return;
      }
      if (quantity <= 0) {
        cout << "Invalid quantity!\n";
        return;
      }
      Product &product = products[productId];
      if (!currentUser->isFollowing(product.getSellerId())) {
        cout << "You can only buy products from users you follow!\n";
        return;
      }
      // make sure enough stock
      if (quantity > product.getStock()) {
        cout << "Only " << product.getStock() << " item(s) available!\n";
        return;
      }
      currentUser->getCart().addProduct(productId, quantity);
      cout << "Added to cart!\n";
    }

    void removeFromCart(int productId)
    {
      if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
      }
      Cart &cart = currentUser->getCart();
      if (!cart.hasProduct(productId)) {
        cout << "Product not found in cart!\n";
        return;
      }
      cart.removeProduct(productId);
      cout << "Product removed from cart!\n";
    }

    void viewCart()
    {
      if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
      }
      Cart &cart = currentUser->getCart();
      if (cart.empty()) {
        cout << "Your cart is empty!\n";
        return;
      }

      cout << "\n===================== CART =====================\n\n";
      cout << left
           << setw(6) << "ID"
           << setw(18) << "Product"
           << setw(10) << "Qty"
           << setw(12) << "Price"
           << setw(12) << "Subtotal" << endl;
      cout << "----------------------------------------------------------\n";

      for (auto item : cart.getItems()) {
        int productId = item.first;
        int quantity = item.second;
        Product &product = products[productId];
        cout << left
             << setw(6) << product.getId()
             << setw(18) << product.getName()
             << setw(10) << quantity
             << setw(12) << product.getPrice()
             << setw(12) << product.getPrice() * quantity
             << endl;
      }
      cout << "\nTotal : Rs. " << cart.getTotalPrice(products) << endl;
    }

    void checkout()
    {
      if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
      }
      Cart &cart = currentUser->getCart();
      if (cart.empty()) {
        cout << "Your cart is empty!\n";
        return;
      }

      // check stock before doing anything
      for (auto item : cart.getItems()) {
        int productId = item.first;
        int quantity = item.second;
        if (quantity > products[productId].getStock()) {
          cout << products[productId].getName() << " does not have enough stock!\n";
          return;
        }
      }

      double total = cart.getTotalPrice(products);
      if (!currentUser->pay(total)) return;

      for (auto item : cart.getItems()) {
        int productId = item.first;
        int quantity = item.second;
        Product &product = products[productId];
        product.decreaseStock(quantity);
        // pay the seller
        users[usernameToIndex[getUsernameById(product.getSellerId())]]
            .addBalance(product.getPrice() * quantity);
      }

      cart.clear();
      cout << "Checkout successful!\n";
    }

    void toggleFollow(string username)
    {
      if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
      }
      if (!usernameToIndex.count(username)) {
        cout << "User does not exist!\n";
        return;
      }
      int index = usernameToIndex[username];
      // cant follow yourself
      if (users[index].getId() == currentUser->getId()) {
        cout << "You cannot follow yourself!\n";
        return;
      }
      int id = users[index].getId();
      if (currentUser->isFollowing(id)) {
        currentUser->unfollow(id);
        cout << "Unfollowed " << username << "!\n";
      } else {
        currentUser->follow(id);
        cout << "Following " << username << "!\n";
      }
    }

    void browseUsers()
    {
      if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
      }
      cout << "\n================== USERS ==================\n\n";
      cout << left << setw(8) << "ID" << setw(20) << "Username" << setw(15) << "Following" << endl;
      cout << "-------------------------------------------\n";
      for (User &user : users) {
        if (user.getId() == currentUser->getId()) continue;
        cout << left
             << setw(8) << user.getId()
             << setw(20) << user.getUsername()
             << setw(15) << (currentUser->isFollowing(user.getId()) ? "Yes" : "No")
             << endl;
      }
      cout << endl;
    }

    void browseProducts()
    {
      if (currentUser == nullptr) {
        cout << "Please login first!\n";
        return;
      }
      cout << "\n==================== PRODUCTS ====================\n\n";
      cout << left
           << setw(6) << "ID"
           << setw(18) << "Product"
           << setw(12) << "Price"
           << setw(10) << "Stock"
           << setw(15) << "Seller" << endl;
      cout << "-------------------------------------------------------------\n";

      bool found = false;
      for (Product &product : products) {
        // only show products from followed sellers
        if (!currentUser->isFollowing(product.getSellerId())) continue;
        cout << left
             << setw(6) << product.getId()
             << setw(18) << product.getName()
             << setw(12) << product.getPrice()
             << setw(10) << product.getStock()
             << setw(15) << getUsernameById(product.getSellerId())
             << endl;
        found = true;
      }
      if (!found) cout << "No products available.\n";
    }
};



// ──────────────────────────────────────────────
//  Helpers
// ──────────────────────────────────────────────

void cls() { cout << "\033[2J\033[1;1H"; }

void hr(char c = '-', int n = 52) { cout << string(n, c) << "\n"; }

void header(string title)
{
    cls();
    hr('=');
    cout << "  [MARKETPLACE]  " << title << "\n";
    hr('=');
    cout << "\n";
}



string getLine(string msg)
{
    string val;
    cout << msg;
    getline(cin, val);
    return val;
}

int getInt(string msg)
{
    string raw;
    cout << msg;
    getline(cin, raw);
    try { return stoi(raw); }
    catch (...) { return -1; }
}

double getDouble(string msg)
{
    string raw;
    cout << msg;
    getline(cin, raw);
    try { return stod(raw); }
    catch (...) { return -1.0; }
}

// ──────────────────────────────────────────────
//  Forward declarations
// ──────────────────────────────────────────────

void screen_home(Marketplace &market);
void screen_main(Marketplace &market);
void screen_users(Marketplace &market);
void screen_products(Marketplace &market);
void screen_cart(Marketplace &market);
void screen_listings(Marketplace &market);

// ──────────────────────────────────────────────
//  HOME  (login / register)
// ──────────────────────────────────────────────

void screen_home(Marketplace &market)
{
    while (true)
    {
        header("Welcome");
        cout << "  1)  Login\n";
        cout << "  2)  Register\n";
        cout << "  3)  Exit\n\n";
        string choice = getLine("  Enter choice: ");

        if (choice == "1")
        {
            cout << "\n";
            string uname = getLine("  Username : ");
            string pwd   = getLine("  Password : ");
            cout << "\n";
            if (market.login(uname, pwd))
            {
                screen_main(market);
            }

        }
        else if (choice == "2")
        {
            cout << "\n";
            string uname = getLine("  Username : ");
            string pwd   = getLine("  Password : ");
            double bal   = getDouble("  Starting balance (Rs.) : ");
            if (bal < 0) {
                cout << "  Invalid amount.\n";

                continue;
            }
            cout << "\n";
            market.registerUser(uname, pwd, bal);

        }
        else if (choice == "3")
        {
            cls();
            cout << "\n  Goodbye!\n\n";
            break;
        }
        else
        {
            cout << "  Invalid option.\n";

        }
    }
}

// ──────────────────────────────────────────────
//  MAIN MENU
// ──────────────────────────────────────────────

void screen_main(Marketplace &market)
{
    while (true)
    {
        header("Main Menu");
        cout << "  1)  Logout\n";
        cout << "  2)  Browse Users     (follow / unfollow)\n";
        cout << "  3)  Browse Products  (add / remove from cart)\n";
        cout << "  4)  View Cart        (checkout)\n";
        cout << "  5)  My Listings      (view & list products)\n\n";
        string choice = getLine("  Enter choice: ");

        if (choice == "1")
        {
            cout << "\n";
            market.logout();

            break;
        }
        else if (choice == "2") { screen_users(market); }
        else if (choice == "3") { screen_products(market); }
        else if (choice == "4") { screen_cart(market); }
        else if (choice == "5") { screen_listings(market); }
        else {
            cout << "  Invalid option.\n";

        }
    }
}

// ──────────────────────────────────────────────
//  USERS PAGE
// ──────────────────────────────────────────────

void screen_users(Marketplace &market)
{
    while (true)
    {
        header("Browse Users");
        market.browseUsers();
        cout << "  Enter a username to follow/unfollow.\n";
        cout << "  Leave blank to go back.\n\n";
        string uname = getLine("  Username: ");
        if (uname.empty()) break;
        cout << "\n";
        market.toggleFollow(uname);

    }
}

// ──────────────────────────────────────────────
//  PRODUCTS PAGE
// ──────────────────────────────────────────────

void screen_products(Marketplace &market)
{
    while (true)
    {
        header("Browse Products");
        market.browseProducts();
        cout << "\n";
        cout << "  a)  Add item to cart\n";
        cout << "  r)  Remove item from cart\n";
        cout << "  b)  Back\n\n";
        string choice = getLine("  Enter choice: ");

        if (choice == "b") break;
        else if (choice == "a")
        {
            cout << "\n";
            int pid = getInt("  Product ID : ");
            int qty = getInt("  Quantity   : ");
            if (pid < 0 || qty < 0) {
                cout << "  Invalid input.\n";

                continue;
            }
            cout << "\n";
            market.addToCart(pid, qty);

        }
        else if (choice == "r")
        {
            cout << "\n";
            int pid = getInt("  Product ID to remove: ");
            if (pid < 0) {
                cout << "  Invalid input.\n";

                continue;
            }
            cout << "\n";
            market.removeFromCart(pid);

        }
        else {
            cout << "  Invalid option.\n";

        }
    }
}

// ──────────────────────────────────────────────
//  CART PAGE
// ──────────────────────────────────────────────

void screen_cart(Marketplace &market)
{
    while (true)
    {
        header("Your Cart");
        market.viewCart();
        cout << "\n";
        cout << "  1)  Checkout\n";
        cout << "  2)  Back\n\n";
        string choice = getLine("  Enter choice: ");

        if (choice == "2") break;
        else if (choice == "1")
        {
            cout << "\n";
            market.checkout();

        }
        else {
            cout << "  Invalid option.\n";

        }
    }
}

// ──────────────────────────────────────────────
//  MY LISTINGS PAGE
// ──────────────────────────────────────────────

void screen_listings(Marketplace &market)
{
    while (true)
    {
        header("My Listings");
        cout << "  1)  View my listed products\n";
        cout << "  2)  List a new product\n";
        cout << "  3)  Change price of a product\n";
        cout << "  4)  Back\n\n";
        string choice = getLine("  Enter choice: ");

        if (choice == "4") break;
        else if (choice == "1")
        {
            cout << "\n";
            market.browseMyListings();

        }
        else if (choice == "2")
        {
            cout << "\n";
            string name  = getLine("  Product name  : ");
            double price = getDouble("  Price (Rs.)   : ");
            int stock    = getInt("  Stock qty     : ");
            if (price < 0 || stock < 0) {
                cout << "  Invalid input.\n";

                continue;
            }
            cout << "\n";
            market.listProduct(name, price, stock);

        }
        else if (choice == "3")
        {
            cout << "\n";
            int pid       = getInt("  Product ID  : ");
            double newprice = getDouble("  New price   : ");
            if (pid < 0 || newprice < 0) {
                cout << "  Invalid input.\n";

                continue;
            }
            cout << "\n";
            market.changePrice(pid, newprice);

        }
        else {
            cout << "  Invalid option.\n";

        }
    }
}

// ──────────────────────────────────────────────
//  ENTRY POINT  (replaces main.cpp's main)
// ──────────────────────────────────────────────

int main()
{

    Marketplace market;

    market.registerUser("shashi", "pass123", 100000);
    market.registerUser("alice", "hello", 50000);
    market.registerUser("bob", "bob123", 75000);
    market.registerUser("charlie", "charlie123", 60000);

    market.login("alice", "hello");
    market.listProduct("Keyboard", 1200, 10);
    market.listProduct("Mouse", 700, 15);
    market.logout();

    market.login("bob", "bob123");
    market.listProduct("Monitor", 15000, 5);
    market.listProduct("Webcam", 2500, 8);
    market.logout();

    market.login("charlie", "charlie123");
    market.listProduct("Headphones", 3000, 6);
    market.logout();
    screen_home(market);
    return 0;
}

Writing main.cpp


In [ ]:
!g++ -std=c++17 main.cpp -o main
!./main

shashi registered successfully!
alice registered successfully!
bob registered successfully!
charlie registered successfully!
Welcome alice!
Keyboard listed successfully!
Mouse listed successfully!
alice logged out!
Welcome bob!
Monitor listed successfully!
Webcam listed successfully!
bob logged out!
Welcome charlie!
Headphones listed successfully!
charlie logged out!
  [MARKETPLACE]  Welcome

  1)  Login
  2)  Register
  3)  Exit

  Enter choice: 1

  Username : shashi
  Password : pass123

Welcome shashi!
  [MARKETPLACE]  Main Menu

  1)  Logout
  2)  Browse Users     (follow / unfollow)
  3)  Browse Products  (add / remove from cart)
  4)  View Cart        (checkout)
  5)  My Listings      (view & list products)

  Enter choice: 2
  [MARKETPLACE]  Browse Users


================== USERS ==================

ID      Username            Following      
-------------------------------------------
2       alice               No             
3       bob                 No             
4   

In [ ]:
1
12
